In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

%matplotlib inline
%load_ext autoreload
%autoreload 2

blackhole_path = Path("/dtu/blackhole/13/213811/s243425/")
root_path_classifier = blackhole_path / "images/IMAGENET128/ddim_classifier_fixed_class10000_train%100_step50_S5_epi_unc_1234"
# root_path = blackhole_path / "images/IMAGENET128/ddim_fixed_class10000_train%100_step50_S5_epi_unc_1234_full"
root_path = blackhole_path  / "images/IMAGENET128/ddim_fixed_class10000_train%100_step50_S5_epi_unc_1234_1024"

In [ ]:
import glob
import os
# samples = os.listdir(blackhole_path / "images/IMAGENET128")
runs_path = glob.glob(str(blackhole_path / "images/IMAGENET128/ddim*full"))
for i, run in enumerate(runs_path):
    print(f"{i}: {run}")

In [ ]:
# selected_runs = [1,2,3]
selected_runs = [0,2]
runs = [] 
for run_id in selected_runs:
    run_path = runs_path[run_id]
    run_name = run_path.split("/")[-1]
    data = {}
    data['clip'] = np.load(Path(run_path) / "entropy_clip.npy")
    data['hpsv3'] = np.load(Path(run_path) / "0" / "hps.npy")
    imgs_path = Path(run_path) / "0" / "imgs"
    globbed_imgs = list(imgs_path.glob("*.png"))
    print(f"Found {len(globbed_imgs)} images.")
    data['imgs'] = globbed_imgs
    data['path'] = run_path
    data['name'] = run_name
    runs.append(data)

In [ ]:
# Top best images by HPSv3
import math

run = runs[0]

# Ensure deterministic alignment with score arrays
globbed_imgs = sorted(imgs_path.glob("*.png"))

top_k = 12
scores = np.asarray(run['hpsv3']).reshape(-1)

if len(globbed_imgs) != len(scores):
    raise ValueError(f"Image count ({len(globbed_imgs)}) and score count ({len(scores)}) do not match.")

top_idx = np.argsort(scores)[-top_k:][::-1]

ncols = 4
nrows = math.ceil(top_k / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
axes = np.array(axes).reshape(-1)

for ax, idx in zip(axes, top_idx):
    img = plt.imread(run['imgs'][idx])
    ax.imshow(img)
    ax.set_title(f"idx={idx} | hpsv3={scores[idx]:.4f}")
    ax.axis("off")

for ax in axes[len(top_idx):]:
    ax.axis("off")

plt.suptitle(f"Top {top_k} images by HPSv3", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Top best images by entropy_clip (lowest values are best)
import math

run = runs[0]

# Ensure deterministic alignment with score arrays
globbed_imgs = sorted(run['imgs'])

top_k = 12
scores = np.asarray(run['clip']).reshape(-1)

if len(globbed_imgs) != len(scores):
    raise ValueError(f"Image count ({len(globbed_imgs)}) and score count ({len(scores)}) do not match.")

# Lower entropy means better
top_idx = np.argsort(scores)[:top_k]

ncols = 4
nrows = math.ceil(top_k / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
axes = np.array(axes).reshape(-1)

for ax, idx in zip(axes, top_idx):
    img = plt.imread(run['imgs'][idx])
    ax.imshow(img)
    ax.set_title(f"idx={idx} | entropy_clip={scores[idx]:.4f}")
    ax.axis("off")

for ax in axes[len(top_idx):]:
    ax.axis("off")

plt.suptitle(f"Top {top_k} images by lowest entropy_clip", fontsize=14)
plt.tight_layout()
plt.show()

## Segmented Correlation: bad vs remaining

Goal: test whether the metrics align more strongly on clearly bad samples than on the rest.

- `hpsv3`: lower is worse
- `entropy_clip`: higher is worse

We define the **worst 20% on both metrics** as:
- `hpsv3 <= 20th percentile`
- `entropy_clip >= 80th percentile`

Then compare correlation in this subset versus all remaining samples.

In [ ]:
# Simple full-data correlation view (no percentile thresholds)
# x = entropy (higher = worse), y = -HPS (higher = worse)

x = np.asarray(ent).reshape(-1)
y = -np.asarray(hps).reshape(-1)

if x.shape[0] != y.shape[0]:
    raise ValueError(f"Mismatched lengths: x={x.shape[0]}, y={y.shape[0]}")

# Correlation summary
pearson_r = np.corrcoef(x, y)[0, 1]
rank_x = np.argsort(np.argsort(x))
rank_y = np.argsort(np.argsort(y))
spearman_rho = np.corrcoef(rank_x, rank_y)[0, 1]

# Best-fit line
coef = np.polyfit(x, y, 1)
x_line = np.linspace(x.min(), x.max(), 200)
y_line = coef[0] * x_line + coef[1]

# Optional trend by equal-count bins (still no thresholding)
n_bins = 15
q = np.linspace(0, 1, n_bins + 1)
edges = np.quantile(x, q)
centers = []
means = []
for i in range(n_bins):
    lo, hi = edges[i], edges[i + 1]
    m = (x >= lo) & (x < hi) if i < n_bins - 1 else (x >= lo) & (x <= hi)
    if m.any():
        centers.append(0.5 * (lo + hi))
        means.append(y[m].mean())

plt.figure(figsize=(7, 6))
plt.scatter(x, y, s=12, alpha=0.22, color="tab:blue", label="Samples")
plt.plot(x_line, y_line, color="tab:red", linewidth=2, label=f"Linear fit (r={pearson_r:.3f})")
plt.plot(centers, means, color="tab:green", marker="o", linewidth=2, label="Binned mean trend")

plt.xlabel("Entropy Clip (higher = worse)")
plt.ylabel("HPS badness (-HPS, higher = worse)")
plt.title("Correlation Between Entropy and HPS Badness")
plt.legend(loc="best")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print(f"Pearson r (linear): {pearson_r:.4f}")
print(f"Spearman rho (rank): {spearman_rho:.4f}")

In [ ]:
# Uncertainty vs HPSv3 distribution
# Using entropy_clip as uncertainty (higher = more uncertain)

uncertainty = np.asarray(entropy_siglip).reshape(-1)
hps = np.asarray(entropy_clip).reshape(-1)

if uncertainty.shape[0] != hps.shape[0]:
    raise ValueError(f"Mismatched lengths: uncertainty={uncertainty.shape[0]}, hps={hps.shape[0]}")

pearson_r = np.corrcoef(uncertainty, hps)[0, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: raw scatter
axes[0].scatter(uncertainty, hps, s=10, alpha=0.25, color="tab:blue")
axes[0].set_xlabel("Uncertainty (Entropy Clip)")
axes[0].set_ylabel("HPSv3")
axes[0].set_title(f"Scatter: Uncertainty vs HPSv3 (r={pearson_r:.3f})")
axes[0].grid(alpha=0.2)

# Right: 2D distribution (hexbin)
hb = axes[1].hexbin(uncertainty, hps, gridsize=45, cmap="viridis", mincnt=1)
axes[1].set_xlabel("Uncertainty (Entropy Clip)")
axes[1].set_ylabel("HPSv3")
axes[1].set_title("Joint Distribution (Hexbin)")
cb = fig.colorbar(hb, ax=axes[1])
cb.set_label("Count")

plt.tight_layout()
plt.show()

print(f"Pearson correlation (uncertainty vs HPSv3): {pearson_r:.4f}")

In [ ]:
# Uncertainty vs HPSv3 distribution
# Using entropy_clip as uncertainty (higher = more uncertain)

run = runs[2]

uncertainty = np.asarray(run['clip']).reshape(-1)
hps = np.asarray(run['hpsv3']).reshape(-1)

if uncertainty.shape[0] != hps.shape[0]:
    raise ValueError(f"Mismatched lengths: uncertainty={uncertainty.shape[0]}, hps={hps.shape[0]}")

pearson_r = np.corrcoef(uncertainty, hps)[0, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: raw scatter
axes[0].scatter(uncertainty, hps, s=10, alpha=0.25, color="tab:blue")
axes[0].set_xlabel("Uncertainty (Entropy Clip)")
axes[0].set_ylabel("HPSv3")
axes[0].set_title(f"Scatter: Uncertainty vs HPSv3 (r={pearson_r:.3f})")
axes[0].grid(alpha=0.2)

# Right: 2D distribution (hexbin)
hb = axes[1].hexbin(uncertainty, hps, gridsize=45, cmap="viridis", mincnt=1)
axes[1].set_xlabel("Uncertainty (Entropy Clip)")
axes[1].set_ylabel("HPSv3")
axes[1].set_title("Joint Distribution (Hexbin)")
cb = fig.colorbar(hb, ax=axes[1])
cb.set_label("Count")

plt.tight_layout()
plt.show()

print(f"Pearson correlation (uncertainty vs HPSv3): {pearson_r:.4f}")

In [ ]:
# Outlier corners: very low-low and very high-high
import math

run = runs[0]

uncertainty = np.asarray(run['clip']).reshape(-1)
hps = np.asarray(run['hpsv3']).reshape(-1)
globbed_imgs = sorted(run['imgs'])
if not (len(uncertainty) == len(hps) == len(globbed_imgs)):
    raise ValueError(
        f"Length mismatch: uncertainty={len(uncertainty)}, hps={len(hps)}, images={len(globbed_imgs)}"
    )

# Strict corner definition by tails
tail_pct = 5
q_low = tail_pct / 100.0
q_high = 1.0 - q_low

u_low, u_high = np.quantile(uncertainty, [q_low, q_high])
h_low, h_high = np.quantile(hps, [q_low, q_high])

low_low_mask = (uncertainty <= u_low) & (hps <= h_low)
high_high_mask = (uncertainty >= u_high) & (hps >= h_high)
low_high_mask = (uncertainty <= u_low) & (hps >= h_high)
high_low_mask = (uncertainty >= u_high) & (hps <= h_low)

idx_low_low_strict = np.where(low_low_mask)[0]
idx_high_high_strict = np.where(high_high_mask)[0]
idx_low_high_strict = np.where(low_high_mask)[0]
idx_high_low_strict = np.where(high_low_mask)[0]

# Fallback: nearest-corner samples when strict intersection is empty
max_show = 12
z_unc = (uncertainty - uncertainty.mean()) / (uncertainty.std() + 1e-12)
z_hps = (hps - hps.mean()) / (hps.std() + 1e-12)
corner_score = z_unc + z_hps

if len(idx_low_low_strict) == 0:
    low_candidates = np.where((uncertainty <= np.quantile(uncertainty, 0.35)) & (hps <= np.quantile(hps, 0.35)))[0]
    if len(low_candidates) == 0:
        low_candidates = np.arange(len(uncertainty))
    idx_low_low = low_candidates[np.argsort(corner_score[low_candidates])[:max_show]]
    low_mode = "fallback-nearest-corner"
else:
    idx_low_low = idx_low_low_strict
    low_mode = "strict-tail-intersection"

if len(idx_high_high_strict) == 0:
    high_candidates = np.where((uncertainty >= np.quantile(uncertainty, 0.65)) & (hps >= np.quantile(hps, 0.65)))[0]
    if len(high_candidates) == 0:
        high_candidates = np.arange(len(uncertainty))
    idx_high_high = high_candidates[np.argsort(-corner_score[high_candidates])[:max_show]]
    high_mode = "fallback-nearest-corner"
else:
    idx_high_high = idx_high_high_strict
    high_mode = "strict-tail-intersection"

if len(idx_low_high_strict) == 0:
    low_high_candidates = np.where((uncertainty <= np.quantile(uncertainty, 0.35)) & (hps >= np.quantile(hps, 0.65)))[0]
    if len(low_high_candidates) == 0:
        low_high_candidates = np.arange(len(uncertainty))
    idx_low_high = low_high_candidates[np.argsort(z_unc[low_high_candidates] - z_hps[low_high_candidates])[:max_show]]
else:
    idx_low_high = idx_low_high_strict

if len(idx_high_low_strict) == 0:
    high_low_candidates = np.where((uncertainty >= np.quantile(uncertainty, 0.65)) & (hps <= np.quantile(hps, 0.35)))[0]
    if len(high_low_candidates) == 0:
        high_low_candidates = np.arange(len(uncertainty))
    idx_high_low = high_low_candidates[np.argsort(z_hps[high_low_candidates] - z_unc[high_low_candidates])[:max_show]]
else:
    idx_high_low = idx_high_low_strict

print(f"tail_pct={tail_pct}%")
print(f"Low-Low strict count (uncertainty<=q{tail_pct}, hps<=q{tail_pct}): {len(idx_low_low_strict)}")
print(f"High-High strict count (uncertainty>=q{100-tail_pct}, hps>=q{100-tail_pct}): {len(idx_high_high_strict)}")
print(f"Low-Low display mode: {low_mode}, shown={min(len(idx_low_low), max_show)}")
print(f"High-High display mode: {high_mode}, shown={min(len(idx_high_high), max_show)}")

# Plot scatter with highlighted corner outliers
plt.figure(figsize=(7, 6))
plt.scatter(uncertainty, hps, s=10, alpha=0.15, color="lightgray", label="All samples")
if len(idx_low_low) > 0:
    plt.scatter(
        uncertainty[idx_low_low],
        hps[idx_low_low],
        s=30,
        alpha=0.85,
        color="tab:blue",
        label=f"Low-Low shown ({len(idx_low_low)})",
    )
if len(idx_high_high) > 0:
    plt.scatter(
        uncertainty[idx_high_high],
        hps[idx_high_high],
        s=30,
        alpha=0.85,
        color="tab:red",
        label=f"High-High shown ({len(idx_high_high)})",
    )
if len(idx_low_high) > 0:
    plt.scatter(
        uncertainty[idx_low_high],
        hps[idx_low_high],
        s=30,
        alpha=0.85,
        color="tab:green",
        label=f"Low-High shown ({len(idx_low_high)})",
    )
if len(idx_high_low) > 0:
    plt.scatter(
        uncertainty[idx_high_low],
        hps[idx_high_low],
        s=30,
        alpha=0.85,
        color="tab:orange",
        label=f"High-Low shown ({len(idx_high_low)})",
    )

plt.axvline(u_low, color="tab:blue", linestyle="--", alpha=0.45)
plt.axhline(h_low, color="tab:blue", linestyle="--", alpha=0.45)
plt.axvline(u_high, color="tab:red", linestyle="--", alpha=0.45)
plt.axhline(h_high, color="tab:red", linestyle="--", alpha=0.45)

plt.xlabel("Uncertainty (Entropy Clip)")
plt.ylabel("HPSv3")
plt.title("Corner Outliers: Low-Low and High-High")
plt.legend(loc="best")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

def show_image_grid(indices, title, descending=False, max_show=12, ncols=4):
    if len(indices) == 0:
        print(f"{title}: no samples found.")
        return

    order = np.argsort(-corner_score[indices]) if descending else np.argsort(corner_score[indices])
    chosen = indices[order][:max_show]

    nrows = math.ceil(len(chosen) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
    axes = np.array(axes).reshape(-1)

    for ax, idx in zip(axes, chosen):
        img = plt.imread(globbed_imgs[idx])
        ax.imshow(img)
        ax.set_title(f"idx={idx} | unc={uncertainty[idx]:.3f} | hps={hps[idx]:.3f}", fontsize=9)
        ax.axis("off")

    for ax in axes[len(chosen):]:
        ax.axis("off")

    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

show_image_grid(
    idx_low_low,
    title=f"Very Low Uncertainty + Very Low HPSv3 (top {min(len(idx_low_low), max_show)})",
    descending=False,
    max_show=max_show,
)

show_image_grid(
    idx_high_high,
    title=f"Very High Uncertainty + Very High HPSv3 (top {min(len(idx_high_high), max_show)})",
    descending=True,
    max_show=max_show,
)

show_image_grid(
    idx_low_high,
    title=f"Low Uncertainty + High HPSv3 (top {min(len(idx_low_high), max_show)})",
    descending=False,
    max_show=max_show,
)

show_image_grid(
    idx_high_low,
    title=f"High Uncertainty + Low HPSv3 (top {min(len(idx_high_low), max_show)})",
    descending=True,
    max_show=max_show,
)

In [ ]:
# Compare the same sample index across 6 model outputs (folders 0..5)
import math

# Set this to any sample index you want to inspect
query_idx = 7626

# Experiment root containing model folders 0..5
compare_root = root_path_classifier  # e.g. .../ddim_classifier_fixed_class10000_train%100_step50_S5_epi_unc_1234
globbed_imgs = list(imgs_path.glob("*.png"))
print(f"Found {len(globbed_imgs)} images.")
def _resolve_image_path(model_img_dir, idx, zero_pad=5):
    # Primary expected naming: 00000.png, 00001.png, ...
    candidate = model_img_dir / f"{idx:0{zero_pad}d}.png"
    if candidate.exists():
        return candidate

    # Fallback: use sorted order if filename convention differs
    files = sorted(model_img_dir.glob("*.png"))
    if idx < 0 or idx >= len(files):
        raise IndexError(f"idx={idx} out of bounds for {model_img_dir} (n={len(files)})")
    return files[idx]

def show_idx_across_models(idx, experiment_root, model_ids=range(6), ncols=3):
    model_ids = list(model_ids)
    n_models = len(model_ids)
    nrows = math.ceil(n_models / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.5 * nrows))
    axes = np.array(axes).reshape(-1)

    for ax, m in zip(axes, model_ids):
        model_dir = experiment_root / str(m)
        img_dir = model_dir / "imgs"
        if not img_dir.exists():
            ax.set_title(f"model {m}: imgs/ not found")
            ax.axis("off")
            continue

        try:
            img_path = _resolve_image_path(img_dir, idx)
            img = plt.imread(img_path)
            ax.imshow(img)

            # Optional HPS annotation if available
            hps_path = model_dir / "hps.npy"
            if hps_path.exists():
                hps_arr = np.load(hps_path).reshape(-1)
                if 0 <= idx < len(hps_arr):
                    ax.set_title(f"model {m} | idx={idx} | hps={hps_arr[idx]:.3f}")
                else:
                    ax.set_title(f"model {m} | idx={idx} | hps=NA")
            else:
                ax.set_title(f"model {m} | idx={idx}")

            ax.axis("off")
        except Exception as e:
            ax.set_title(f"model {m} error")
            ax.text(0.02, 0.5, str(e), fontsize=9, va="center", wrap=True)
            ax.axis("off")

    for ax in axes[n_models:]:
        ax.axis("off")

    plt.suptitle(f"Same sample across models 0..{max(model_ids)} | idx={idx}", fontsize=15)
    plt.tight_layout()
    plt.show()

show_idx_across_models(query_idx, compare_root, model_ids=range(6), ncols=3)

In [ ]:
show_idx_across_models(388, compare_root, model_ids=range(6), ncols=3)

In [ ]:
show_idx_across_models(11014, compare_root, model_ids=range(6), ncols=3)

In [ ]:

show_idx_across_models(4251, compare_root, model_ids=range(6), ncols=3)

In [ ]:
np.corrcoef(entropy_clip, hpsv3)

In [ ]:
hpsv3_sigma.shape

In [ ]:
plt.hist(hpsv3, bins=100)

In [ ]:
import numpy as np
from scipy.integrate import trapezoid as trapz 
import matplotlib.pyplot as plt

# 1. Define "Error" (Inverting HPSv3 so lower is better)
error = np.max(hpsv3) - hpsv3
n_samples = len(hpsv3)
rejection_rates = np.arange(n_samples) / n_samples

# --- LINE 1: Generative Uncertainty Rejection ---
sorted_indices = np.argsort(entropy_clip)[::-1] # Sort by entropy (High to Low)
sorted_error_unc = error[sorted_indices]
average_errors_unc = np.zeros(n_samples)

for i in range(n_samples):
    average_errors_unc[i] = np.mean(sorted_error_unc[i:])
aurc_unc = trapz(average_errors_unc, rejection_rates)

# --- LINE 2: Random Rejection Baseline ---
# Average error stays roughly the same as the dataset mean
average_errors_rand = np.full(n_samples, np.mean(error))
aurc_rand = trapz(average_errors_rand, rejection_rates)

# --- LINE 3: Optimal Rejection (The Oracle) ---
# A perfect metric perfectly sorts by the actual error
sorted_error_opt = np.sort(error)[::-1] # Sort directly by error (High to Low)
average_errors_opt = np.zeros(n_samples)

for i in range(n_samples):
    average_errors_opt[i] = np.mean(sorted_error_opt[i:])
aurc_opt = trapz(average_errors_opt, rejection_rates)

# --- LINE 4 : Rarity Rejection (Optional) ---
sorted_indices_rarity = np.argsort(rarity)[::-1] # Sort by rarity (High to Low)
sorted_error_rarity = error[sorted_indices_rarity]
average_errors_rarity = np.zeros(n_samples)

for i in range(n_samples):
    average_errors_rarity[i] = np.mean(sorted_error_rarity[i:])
aurc_rarity = trapz(average_errors_rarity, rejection_rates)

# --- LINE 5 : Realism Rejection (Optional) ---
sorted_indices_realism = np.argsort(-realism)[::-1] # Sort by realism (High to Low)
sorted_error_realism = error[sorted_indices_realism]
average_errors_realism = np.zeros(n_samples)

for i in range(n_samples):
    average_errors_realism[i] = np.mean(sorted_error_realism[i:])
aurc_realism = trapz(average_errors_realism, rejection_rates)

# --- LINE 6 : DINOv2 Entropy Rejection (Optional) ---
sorted_indices_dino = np.argsort(entropy_openclip_h14)[::-1] # Sort by DINOv2 entropy (High to Low)
sorted_error_dino = error[sorted_indices_dino]
average_errors_dino = np.zeros(n_samples)

for i in range(n_samples):
     average_errors_dino[i] = np.mean(sorted_error_dino[i:])
aurc_dino = trapz(average_errors_dino, rejection_rates)


# --- Plotting the Results ---
plt.figure(figsize=(10, 6))

# Plot the three lines
plt.plot(rejection_rates * 100, average_errors_rand, 
         label=f'Random Baseline (AURC={aurc_rand:.2f})', 
         linestyle='--', color='gray', linewidth=2)

plt.plot(rejection_rates * 100, average_errors_unc, 
         label=f'Generative Uncertainty (AURC={aurc_unc:.2f})', 
         color='blue', linewidth=2.5)

plt.plot(rejection_rates * 100, average_errors_opt, 
         label=f'Optimal Oracle (AURC={aurc_opt:.2f})', 
         linestyle=':', color='green', linewidth=2.5)

plt.plot(rejection_rates * 100, average_errors_rarity,
         label=f'Rarity (AURC={aurc_rarity:.2f})',
         linestyle='-.', color='orange', linewidth=2.5)

plt.plot(rejection_rates * 100, average_errors_realism,
         label=f'Realism (AURC={aurc_realism:.2f})',
         linestyle='--', color='purple', linewidth=2.5)

plt.plot(rejection_rates * 100, average_errors_dino,
         label=f'DINOv2 Entropy (AURC={aurc_dino:.2f})',
         linestyle='-.', color='red', linewidth=2.5)

# Formatting
plt.xlabel('Rejection Rate (%)')
plt.ylabel('Average Error of Remaining Images')
plt.title('Rejection Curve: Generative Uncertainty vs Optimal Bound')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlim(0, 100)

plt.show()

In [ ]:
plt.hist(rarity[rarity != np.inf], bins=100)

In [ ]:
run

In [ ]:
(rarity == np.inf).sum()

In [ ]:
np.corrcoef(entropy_dinov2, hpsv3)

In [ ]:
features_dinov2 = torsh.load("/dtu/blackhole/13/213811/s243425/images/IMAGENET128/ddim_fixed_class10000_train%100_step50_S5_epi_unc_1234_full/0/dinov2_vits14_reg_features.pt")

In [ ]:
import numpy as np
from scipy.integrate import trapezoid as trapz 
import matplotlib.pyplot as plt

# 1. Define "Error" (Inverting HPSv3 so lower is better)
error = np.max(runs[0]['hpsv3']) - runs[0]['hpsv3']
n_samples = len(runs[0]['hpsv3'])
rejection_rates = np.arange(n_samples) / n_samples

plt.figure(figsize=(10, 6))

# Average error stays roughly the same as the dataset mean
average_errors_rand = np.full(n_samples, np.mean(error))
aurc_rand = trapz(average_errors_rand, rejection_rates)

plt.plot(rejection_rates * 100, average_errors_rand, 
         label=f'Random Baseline (AURC={aurc_rand:.2f})', 
         linestyle='--', color='gray', linewidth=2)

for run in runs:

    sorted_indices = np.argsort(run['clip'])[::-1] # Sort by entropy (High to Low)
    sorted_error_unc = error[sorted_indices]
    average_errors_unc = np.zeros(n_samples)

    for i in range(n_samples):
        average_errors_unc[i] = np.mean(sorted_error_unc[i:])
    aurc_unc = trapz(average_errors_unc, rejection_rates)

    plt.plot(rejection_rates * 100, average_errors_unc, 
             label=f'{run['name']} (AURC={aurc_unc:.2f})', 
             linewidth=2.5)

# --- LINE 3: Optimal Rejection (The Oracle) ---
# A perfect metric perfectly sorts by the actual error
sorted_error_opt = np.sort(error)[::-1] # Sort directly by error (High to Low)
average_errors_opt = np.zeros(n_samples)

for i in range(n_samples):
    average_errors_opt[i] = np.mean(sorted_error_opt[i:])
aurc_opt = trapz(average_errors_opt, rejection_rates)

plt.plot(rejection_rates * 100, average_errors_opt, 
         label=f'Optimal Oracle (AURC={aurc_opt:.2f})', 
         linestyle=':', color='green', linewidth=2.5)


# Formatting
plt.xlabel('Rejection Rate (%)')
plt.ylabel('Average Error of Remaining Images')
plt.title('Rejection Curve: Generative Uncertainty vs Optimal Bound')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlim(0, 100)

plt.show()